# SDF Reproduction: F-INR vs Baseline

This notebook runs a representative SDF experiment on the bundled asset `assets/sdf/voxels_sdfs_200_armadillo.npy`.

To keep the comparison practical and reproducible, the public configs downsample the grid before training and compare the clean factorized SDF model against the clean baseline coordinate MLP on the same reconstruction problem.

In [ ]:
import importlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for entry in (ROOT, ROOT / "scripts", ROOT / "src"):
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

import f_inr_sdf  # noqa: E402

importlib.reload(f_inr_sdf)
run_sdf_experiment = f_inr_sdf.run_sdf_experiment
from finr_shape import downsample_sdf_grid, load_sdf_grid, truncate_sdf  # noqa: E402

finr_config = json.load((ROOT / "configs/sdf/armadillo_finr.json").open())
baseline_config = json.load((ROOT / "configs/sdf/armadillo_baseline.json").open())
print(f_inr_sdf.__file__)
grid = load_sdf_grid(ROOT / finr_config["sdf_path"])
grid = downsample_sdf_grid(grid, stride=finr_config["downsample_stride"])
grid = truncate_sdf(grid, truncation=finr_config["truncate"])
grid.shape

In [ ]:
mid = grid.shape[2] // 2
plt.figure(figsize=(5, 5))
plt.imshow(grid[:, :, mid], cmap="coolwarm")
plt.colorbar()
plt.title(f"Asset slice: {Path(finr_config['sdf_path'])}")
plt.axis("off")
plt.show()

In [ ]:
finr_summary, _ = run_sdf_experiment(finr_config)


In [ ]:
baseline_summary, _ = run_sdf_experiment(baseline_config)
summaries = [finr_summary, baseline_summary]
print(json.dumps(summaries, indent=2))
print(
    "Speedup (baseline / F-INR):",
    round(baseline_summary["elapsed_seconds"] / finr_summary["elapsed_seconds"], 3),
)

In [ ]:
finr_recon = np.load(Path(finr_summary["output_dir"]) / "reconstruction.npy")
baseline_recon = np.load(Path(baseline_summary["output_dir"]) / "reconstruction.npy")
mid = finr_recon.shape[2] // 2
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(grid[:, :, mid], cmap="coolwarm")
axes[0].set_title("Input slice")
axes[1].imshow(finr_recon[:, :, mid], cmap="coolwarm")
axes[1].set_title("F-INR slice")
axes[2].imshow(baseline_recon[:, :, mid], cmap="coolwarm")
axes[2].set_title("Baseline slice")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()